In [1]:
import festim as F
from foam2dolfinx import OpenFOAMReader, find_closest_value
from dolfinx.io import gmsh
from mpi4py import MPI
import numpy as np


from scifem import assemble_scalar
from dolfinx import fem
from dolfinx.io import VTXWriter, XDMFFile
import ufl
from dolfinx import cpp as _cpp
from dolfinx.log import set_log_level, LogLevel
from dolfinx.io import gmsh as gmshio
from mpi4py import MPI
from basix.ufl import element
import h_transport_materials as htm

/home/kaelyn/anaconda3/envs/hx-env/lib/python3.12/site-packages/pybtex/plugin/__init__.py:26: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [ ]:
## Reading OpenFOAM fields using foam2dolfinx
my_reader = OpenFOAMReader(
    filename="/home/kaelyn/Fusion-Heat-Exchangers/openfoam/steadyThickHX/hx.foam",
    cell_type=10,
)

In [3]:
facet_tags = my_reader.create_facet_meshtags()
cell_tags = my_reader.create_cell_meshtags()

mesh = my_reader.dolfinx_meshes_dict["_global"]  # single-domain: use "default"

Boundary patch summary:
  bc_inner_outlet: id=1, n_facets=36
  bc_inner_inlet: id=2, n_facets=36
  fluid_1_tubeside_to_solid_shell: id=3, n_facets=0
  fluid_1_tubeside_to_solid_baffles: id=4, n_facets=0
  fluid_1_tubeside_to_solid_pipes: id=5, n_facets=0
  bc_outer_outlet: id=6, n_facets=34
  bc_outer_inlet: id=7, n_facets=34
  fluid_2_shellside_to_solid_baffles: id=8, n_facets=0
  fluid_2_shellside_to_solid_shell: id=9, n_facets=0
  fluid_2_shellside_to_solid_pipes: id=10, n_facets=0
  solid_baffles_to_fluid_2_shellside: id=11, n_facets=0
  solid_baffles_to_fluid_1_tubeside: id=12, n_facets=0
  solid_baffles_to_solid_pipes: id=13, n_facets=0
  solid_baffles_to_solid_shell: id=14, n_facets=0
  solid_pipes_to_solid_baffles: id=15, n_facets=0
  solid_pipes_to_fluid_1_tubeside: id=16, n_facets=0
  solid_pipes_to_fluid_2_shellside: id=17, n_facets=0
  shell_outer_wall: id=18, n_facets=2466
  solid_shell_to_fluid_1_tubeside: id=19, n_facets=0
  solid_shell_to_fluid_2_shellside: id=20, n_fac

In [4]:
## Defining material properties
inner_fluid_mat = F.Material(
    D_0=1e-3, E_D=0, K_S_0=10, E_K_S=0
)  # fluid_1_tubeside (coolant)
outer_fluid_mat = F.Material(
    D_0=1e-3, E_D=0, K_S_0=10, E_K_S=0
)  # fluid_2_shellside (breeder)
shell_mat = F.Material(D_0=1e-4, E_D=0, K_S_0=15, E_K_S=0)  # solid_shell
pipes_mat = F.Material(D_0=1e-4, E_D=0, K_S_0=15, E_K_S=0)  # solid_pipes
baffles_mat = F.Material(D_0=1e-4, E_D=0, K_S_0=15, E_K_S=0)  # solid_baffles
# shell_mat       = F.Material(D_0=1e-4, E_D=0, K_S_0=5,  E_K_S=0)  # solid_shell
# pipes_mat       = F.Material(D_0=1e-4, E_D=0, K_S_0=5,  E_K_S=0)  # solid_pipes
# baffles_mat     = F.Material(D_0=1e-4, E_D=0, K_S_0=5,  E_K_S=0)  # solid_baffles

In [5]:
import ufl


class SurfaceAdvectionFlux(F.SurfaceFlux):
    """Computes the advection flux of a field on a given surface

    Args:
        field (festim.Species): species for which the surface flux is computed
        surface (festim.SurfaceSubdomain1D): surface subdomain
        filename (str, optional): name of the file to which the surface flux is exported

    Attributes:
        see `festim.SurfaceFlux`
    """

    def __init__(self, field, surface, filename, velocity_field):

        super().__init__(field=field, surface=surface, filename=filename)
        self.velocity_field = velocity_field

    @property
    def title(self):
        return f"{self.field.name} advection flux surface {self.surface.id}"

    def compute(self, u, ds: ufl.Measure, entity_maps=None):
        if isinstance(u, ufl.indexed.Indexed):
            mesh = self.field.sub_function_space.mesh
        else:
            mesh = u.function_space.mesh

        n = ufl.FacetNormal(mesh)

        # Diffusive flux
        surface_flux = assemble_scalar(
            fem.form(
                -self.D * ufl.dot(ufl.grad(u), n) * ds(self.surface.id),
                entity_maps=entity_maps,
            )
        )

        # Advective flux — interpolate velocity onto the submesh first
        from dolfinx.fem import Function, functionspace

        vel_space = functionspace(
            mesh, self.velocity_field.function_space.ufl_element()
        )
        vel_local = Function(vel_space)
        vel_local.interpolate(self.velocity_field)

        advective_flux = assemble_scalar(
            fem.form(
                u * ufl.inner(vel_local, n) * ds(self.surface.id),
                entity_maps=entity_maps,
            )
        )

        self.value = surface_flux + advective_flux
        self.data.append(self.value)

In [6]:
## Volume subdomains — use cell zone IDs from summary
inner_fluid_vol = F.VolumeSubdomain(id=1, material=inner_fluid_mat)  # fluid_1_tubeside
outer_fluid_vol = F.VolumeSubdomain(id=2, material=outer_fluid_mat)  # fluid_2_shellside
baffles_vol = F.VolumeSubdomain(id=3, material=baffles_mat)  # solid_baffles
pipes_vol = F.VolumeSubdomain(id=4, material=pipes_mat)  # solid_pipes
shell_vol = F.VolumeSubdomain(id=5, material=shell_mat)  # solid_shell

## Surface subdomains — use boundary patch IDs from summary
inner_outlet = F.SurfaceSubdomain(id=1)
inner_inlet = F.SurfaceSubdomain(id=2)
outer_outlet = F.SurfaceSubdomain(id=6)
outer_inlet = F.SurfaceSubdomain(id=7)
shell_wall = F.SurfaceSubdomain(id=18)
## Defining hydrogen transport problem
my_model = F.HydrogenTransportProblemDiscontinuous()
my_model.mesh = F.Mesh(mesh)
my_model.facet_meshtags = facet_tags
my_model.volume_meshtags = cell_tags

my_model.subdomains = [
    inner_inlet,
    inner_outlet,
    outer_inlet,
    outer_outlet,
    shell_wall,
    shell_vol,
    pipes_vol,
    baffles_vol,
    inner_fluid_vol,
    outer_fluid_vol,
]

my_model.surface_to_volume = {
    inner_inlet: inner_fluid_vol,
    inner_outlet: inner_fluid_vol,
    outer_inlet: outer_fluid_vol,
    outer_outlet: outer_fluid_vol,
    shell_wall: shell_vol,
}

my_model.interfaces = [
    F.Interface(id=22, subdomains=[outer_fluid_vol, shell_vol], penalty_term=1e5),
    F.Interface(id=23, subdomains=[outer_fluid_vol, pipes_vol], penalty_term=1e5),
    F.Interface(id=24, subdomains=[outer_fluid_vol, baffles_vol], penalty_term=1e5),
    F.Interface(id=25, subdomains=[baffles_vol, shell_vol], penalty_term=1e5),
    F.Interface(id=26, subdomains=[inner_fluid_vol, pipes_vol], penalty_term=1e5),
    F.Interface(id=27, subdomains=[inner_fluid_vol, shell_vol], penalty_term=1e5),
    F.Interface(id=28, subdomains=[inner_fluid_vol, baffles_vol], penalty_term=1e5),
    F.Interface(id=29, subdomains=[baffles_vol, pipes_vol], penalty_term=1e5),
]

H = F.Species(
    "H",
    subdomains=[inner_fluid_vol, outer_fluid_vol, shell_vol, pipes_vol, baffles_vol],
)
my_model.species = [H]

In [7]:
my_model.temperature = 400

In [9]:
my_model.boundary_conditions = [
    F.FixedConcentrationBC(subdomain=outer_inlet, value=0, species=H),
    F.FixedConcentrationBC(subdomain=inner_inlet, value=2, species=H),
]

vel_inner = my_reader.create_dolfinx_function_with_cell_data(
    t=2080, name="U", subdomain="fluid_1_tubeside"
)
vel_outer = my_reader.create_dolfinx_function_with_cell_data(
    t=2080, name="U", subdomain="fluid_2_shellside"
)

my_model.advection_terms = [
    F.AdvectionTerm(velocity=vel_inner, subdomain=inner_fluid_vol, species=H),
    F.AdvectionTerm(velocity=vel_outer, subdomain=outer_fluid_vol, species=H),
]

In [11]:
my_model.exports = [
    F.VTXSpeciesExport(
        filename="steady_w_advection_2/inner_fluid.bp",
        field=H,
        subdomain=inner_fluid_vol,
    ),
    F.VTXSpeciesExport(
        filename="steady_w_advection_2/outer_fluid.bp",
        field=H,
        subdomain=outer_fluid_vol,
    ),
    F.VTXSpeciesExport(
        filename="steady_w_advection_2/shell.bp", field=H, subdomain=shell_vol
    ),
    F.VTXSpeciesExport(
        filename="steady_w_advection_2/pipes.bp", field=H, subdomain=pipes_vol
    ),
    F.VTXSpeciesExport(
        filename="steady_w_advection_2/baffles.bp", field=H, subdomain=baffles_vol
    ),
    # outlet_advective_flux,
    # total_shell,
    # total_pipes,
    # total_baffles,
]

my_model.settings = F.Settings(atol=1e-8, rtol=1e-10, transient=False)
my_model.petsc_options = {
    "ksp_type": "preonly",
    "pc_type": "lu",
    "pc_factor_mat_solver_type": "mumps",
}

import dolfinx

dolfinx.log.set_log_level(dolfinx.log.LogLevel.INFO)
my_model.initialise()
my_model.run()

[2026-07-10 14:54:32.840] [info] Requesting connectivity (3, 0) - (0, 0)
[2026-07-10 14:54:32.842] [info] Requesting connectivity (3, 0) - (3, 0)
[2026-07-10 14:54:32.842] [info] Requesting connectivity (3, 0) - (3, 0)
[2026-07-10 14:54:32.845] [info] Requesting connectivity (3, 0) - (3, 0)
[2026-07-10 14:54:32.845] [info] Computing mesh entities of dimension 2
[2026-07-10 14:54:32.848] [info] Computing communication graph edges (using NBX algorithm). Number of input edges: 0
[2026-07-10 14:54:32.848] [info] Finished graph edge discovery using NBX algorithm. Number of discovered edges 0
[2026-07-10 14:54:32.849] [info] Requesting connectivity (2, 0) - (0, 0)
[2026-07-10 14:54:32.849] [info] Requesting connectivity (3, 0) - (2, 0)
[2026-07-10 14:54:32.849] [info] Requesting connectivity (2, 0) - (0, 0)
[2026-07-10 14:54:32.849] [info] Requesting connectivity (2, 0) - (3, 0)
[2026-07-10 14:54:32.861] [info] Requesting connectivity (3, 0) - (0, 0)
[2026-07-10 14:54:32.883] [info] Requesti

In [16]:
import dolfinx.fem as fem
import ufl
from dolfinx.fem.petsc import assemble_vector
from petsc4py import PETSc

for vol, name in [
    (shell_vol, "solid_shell"),
    (pipes_vol, "solid_pipes"),
    (baffles_vol, "solid_baffles"),
]:
    u = H.subdomain_to_post_processing_solution[vol]
    dx = ufl.Measure("dx", domain=vol.submesh)
    total = fem.assemble_scalar(fem.form(u * dx))
    print(f"Total H in {name}: {total:.6e} mol")

Total H in solid_shell: 2.298761e-03 mol
Total H in solid_pipes: 3.886496e-04 mol
Total H in solid_baffles: 6.951542e-04 mol
